# 01 - Data Exploration: Sample Solar DER Data

This notebook demonstrates how to load, explore, and visualise the sample
solar generation data included with the DERIM middleware project.

## Objectives

1. Load the CSV file containing 30 days of synthetic 15-minute solar inverter telemetry.
2. Inspect the data schema and summary statistics.
3. Visualise daily generation profiles, irradiance patterns, and energy accumulation.
4. Identify data quality issues (missing values, outliers).

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['figure.dpi'] = 100

## 1. Load the Sample Data

In [ ]:
df = pd.read_csv('../data/sample_solar_data.csv', parse_dates=['timestamp'])
print(f'Shape: {df.shape}')
print(f'Date range: {df.timestamp.min()} to {df.timestamp.max()}')
df.head(10)

## 2. Summary Statistics

In [ ]:
df.describe().round(3)

In [ ]:
# Check for missing values
print('Missing values per column:')
print(df.isnull().sum())
print(f'\nTotal rows: {len(df)}')
print(f'Unique devices: {df.device_id.nunique()}')

## 3. Power Generation Profile

Visualise the full 30-day power output and zoom into a single week.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=False)

# Full 30-day profile
axes[0].plot(df['timestamp'], df['power_kw'], linewidth=0.5, color='#FF6B00')
axes[0].set_title('Solar Power Generation - Full 30 Days', fontsize=13)
axes[0].set_ylabel('Power (kW)')
axes[0].fill_between(df['timestamp'], df['power_kw'], alpha=0.3, color='#FF6B00')

# One-week zoom
week = df[df['timestamp'] < df['timestamp'].min() + pd.Timedelta(days=7)]
axes[1].plot(week['timestamp'], week['power_kw'], linewidth=0.8, color='#0066CC')
axes[1].set_title('Solar Power Generation - First Week', fontsize=13)
axes[1].set_ylabel('Power (kW)')
axes[1].set_xlabel('Time')
axes[1].fill_between(week['timestamp'], week['power_kw'], alpha=0.3, color='#0066CC')

plt.tight_layout()
plt.show()

## 4. Irradiance vs Power Correlation

In [ ]:
daytime = df[df['irradiance_w_m2'] > 10].copy()

fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(
    daytime['irradiance_w_m2'], daytime['power_kw'],
    c=daytime['panel_temperature_c'], cmap='YlOrRd',
    alpha=0.5, s=10, edgecolors='none'
)
plt.colorbar(scatter, label='Panel Temperature (°C)')
ax.set_xlabel('Irradiance (W/m²)')
ax.set_ylabel('Power Output (kW)')
ax.set_title('Irradiance vs Power Output (coloured by panel temperature)')
plt.tight_layout()
plt.show()

corr = daytime[['power_kw', 'irradiance_w_m2', 'panel_temperature_c']].corr()
print('Correlation matrix (daytime only):')
corr.round(3)

## 5. Daily Energy Production

In [ ]:
df['date'] = df['timestamp'].dt.date
daily_energy = df.groupby('date')['power_kw'].sum() * 0.25  # kWh (15-min intervals)

fig, ax = plt.subplots(figsize=(12, 5))
daily_energy.plot(kind='bar', ax=ax, color='#2ECC71', edgecolor='#27AE60')
ax.set_title('Daily Solar Energy Production', fontsize=13)
ax.set_ylabel('Energy (kWh)')
ax.set_xlabel('Date')
ax.axhline(daily_energy.mean(), color='red', linestyle='--', label=f'Mean: {daily_energy.mean():.1f} kWh')
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print(f'Total energy over 30 days: {daily_energy.sum():.1f} kWh')
print(f'Average daily production:  {daily_energy.mean():.1f} kWh')
print(f'Peak daily production:     {daily_energy.max():.1f} kWh')

## 6. Voltage and Frequency Stability

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['voltage_v'], bins=50, color='#3498DB', edgecolor='white')
axes[0].axvline(230, color='red', linestyle='--', label='Nominal (230V)')
axes[0].set_title('Voltage Distribution')
axes[0].set_xlabel('Voltage (V)')
axes[0].legend()

axes[1].hist(df['frequency_hz'], bins=50, color='#9B59B6', edgecolor='white')
axes[1].axvline(50, color='red', linestyle='--', label='Nominal (50 Hz)')
axes[1].set_title('Frequency Distribution')
axes[1].set_xlabel('Frequency (Hz)')
axes[1].legend()

plt.tight_layout()
plt.show()

## 7. Average Daily Profile (Hourly)

Compute the average power output for each hour of the day across all 30 days.

In [ ]:
df['hour'] = df['timestamp'].dt.hour + df['timestamp'].dt.minute / 60
hourly_avg = df.groupby(df['hour'].round(1))['power_kw'].agg(['mean', 'std'])

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(hourly_avg.index, hourly_avg['mean'], color='#E74C3C', linewidth=2)
ax.fill_between(
    hourly_avg.index,
    hourly_avg['mean'] - hourly_avg['std'],
    hourly_avg['mean'] + hourly_avg['std'],
    alpha=0.2, color='#E74C3C', label='±1 Std Dev'
)
ax.set_title('Average Daily Solar Generation Profile', fontsize=13)
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Power (kW)')
ax.set_xlim(0, 24)
ax.legend()
plt.tight_layout()
plt.show()

## Summary

This exploration confirms that the sample data exhibits realistic solar generation
patterns: a bell-shaped daily profile, strong correlation between irradiance and
power output, and stable grid voltage/frequency. The data is ready for use in
model training (see Notebook 03).